# CIS I — Assignment 1: Pivot Calibration

This notebook walks through the pivot-calibration scenario from the
assignment (Figure 1): a pointer and a calibration object, each
carrying a marker body, observed by an optical tracker. Your job is to
**design** the marker-sphere layouts and workspace placements, then
**calibrate** the pointer's tip position `p_tip` to within about 2mm.

Everything runs against `src.simulation`, a self-contained simulator that plays the role of the physical tracker +
hand-guided robot + calibration sensor described in the handout. It injects measurement noise — your code never sees ground truth
except where explicitly noted for **validation/reporting** in Step 8 (real
hardware wouldn't give you that; the point there is to report your achieved
accuracy, not to use the ground truth in your calibration math itself).



In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
from src import simulation
from src import vct3, Rot, Frame

np.set_printoptions(precision=3, suppress=True)


## Step 1: Design the pointer's marker body

Choose approximate local marker-sphere positions `a_ptr,k` (a `List[vct3]`,
relative to the pointer's own coordinate frame) and the pointer tip's
approximate local position `p_tip_approx`. These are your
*nominal design values* — the simulator will independently perturb the
*actual* manufactured positions by a small, noise amount (bounded at
2mm).


In [2]:
ptr_local_positions = [
    vct3(-50, -50, -50),
    vct3( 50, -50,  50),
    vct3(-50,  50,  50),
    vct3( 50,  50, -50),
]
ptr_tip_approx = vct3(0, 0, 150)  # TODO: vct3, the tip's approximate position in the pointer's local frame



## Step 1.1: Design the calibration object's marker body

Same idea, for the calibration object's local marker positions `a_cal,k`.


In [3]:
cal_local_positions = [
    # TODO: at least 4 non-coplanar vct3 points
    vct3(-40, -40, -40),
    vct3( 40, -40,  40),
    vct3(-40,  40,  40),
    vct3( 40,  40, -40),
    
]


## Step 2: Approximate workspace poses

Specify approximate poses (as `Frame` class from src) for the tracking camera
(`F_tracker`) and the calibration object (`F_cal`), relative to the
workspace. The tracker's usable volume is the tetrahedral region in the
handout's Figure 2 (roughly 100-200cm in front of the tracker, widening
from ~120cm to ~200cm across) — place the calibration object, and later
the pointer, so they're inside that volume.


In [4]:
F_tracker_approx = Frame.eye()  # TODO: Frame, tracker pose relative to the workspace
F_cal_approx = Frame(
    Rot.eye(),
    vct3(0, 0, 1500)
)      # TODO: Frame, calibration object pose relative to the workspace


## Step 3: Build the simulation

Fully provided — this just wires up your Step 1/2 design choices through
`simulation`'s `DefineMarkerBody` / `PlaceMarkerBody` / `CreateTracker`
equivalents. `ptr_placed`/`cal_placed` are `PlacedBody` handles: each pairs
a `MarkerBody` with its hidden actual placement, and gets passed to
`simulation.sample_marker_bodies(...)` whenever you want a reading.


In [5]:
ptr_mb = simulation.define_marker_body("ptr_mb", ptr_local_positions)
cal_mb = simulation.define_marker_body("cal_mb", cal_local_positions)

cal_placed = simulation.place_marker_body(cal_mb, F_cal_approx)
# The pointer's own body doesn't have a meaningful "workspace placement" yet
# It moves every time it's hand-guided (Step 4-6). 
# precision="precise" matches a robot-controlled pose rather than a coarsely, manually placed
# static object (see place_marker_body's docstring).
ptr_placed = simulation.place_marker_body(ptr_mb, Frame.eye(), precision="precise")

ptr = simulation.define_pointer("ptr1", ptr_tip_approx, ptr_mb)
trk = simulation.create_tracker("trk1", F_tracker_approx)

print("simulation built:", ptr_mb.name, cal_mb.name, ptr.name, trk.name)


simulation built: ptr_mb cal_mb ptr1 trk1


## Step 3.1: Refine `a_ptr,k` / `a_cal,k`

> "Observations of marker positions can be used to determine accurate
> estimates of relative positions a_ptr,k of markers on pointer and
> calibration objects... **Hint**: You probably want to take multiple
> readings in order to reduce the uncertainty associated with the a_k
> values."


In [6]:
def refine_marker_body(trk, placed, n_readings=30, passes=3):
    """
    TODO: implement.

    input:
        trk - Tracker to read with
        placed - PlacedBody whose body's nominal marker positions should be
            refined in place (via body.set_marker_positions(...))
        n_readings - int, readings to average per pass
        passes - int, how many times to repeat
    output: None (mutates placed.body's nominal_marker_positions in place)
    """

    for _ in range(passes):

        local_estimates = [
            [] for _ in placed.body.nominal_marker_positions
        ]

        for _ in range(n_readings):

            obs = simulation.sample_marker_bodies(
                trk,
                [placed]
            )

            observed_body = obs[placed.body.name]

            F = observed_body.frame
            F_inv = F.inv()

            for k, marker in enumerate(observed_body.markers):

                a_est = F_inv * marker.p

                local_estimates[k].append(a_est)

        refined_positions = []

        for estimates in local_estimates:

            x = np.mean([p.x for p in estimates])
            y = np.mean([p.y for p in estimates])
            z = np.mean([p.z for p in estimates])

            refined_positions.append(
                vct3(x, y, z)
            )

        placed.body.set_marker_positions(refined_positions)
    
refine_marker_body(trk, ptr_placed)
refine_marker_body(trk, cal_placed)


## Steps 4-6: Collect pivot-calibration observations

For each observation `j`: hand-guide the pointer near the calibration
object at some desired orientation `R_desired,j`, take one simultaneous
tracker reading of *both* bodies, and read the calibration sensor. Loop
skeleton is provided; you choose how many observations to take and how to
vary `R_desired,j`


In [7]:
N_OBSERVATIONS = 30  # TODO

F_ptr_list = []
F_cal_list = []
sensed_tip_list = []
for j in range(N_OBSERVATIONS):
    rx = (j % 5 - 2) * np.pi / 6
    ry = ((j // 5) % 3 - 1) * np.pi / 6
    rz = (j // 15) * np.pi / 6

    R_desired = Rot.xyz(rx, ry, rz)  # TODO: vary the desired pointer orientation across observations
    hg = simulation.hand_guide_pointer(ptr, ptr_placed, cal_placed, R_desired)
    obs = simulation.sample_marker_bodies(trk, [ptr_placed, cal_placed])
    sensed = simulation.sense_tip(ptr, hg)

    F_ptr_list.append(obs["ptr_mb"].frame)
    F_cal_list.append(obs["cal_mb"].frame)
    sensed_tip_list.append(sensed)

print(f"collected {len(F_ptr_list)} observations")


collected 30 observations


`sensed_tip_list` holds each observation's high-accuracy calibration-sensor
reading (or `None` if that observation ended up outside the sensor's 5mm
range) — useful as an independent sanity check on your calibration later,
since it measures the tip position a completely different way (directly,
rather than through the pointer's marker geometry).


## Step 7: Compute a calibrated value for `p_tip`

This is the assignment's core deliverable — implement it yourself and specify the details of implementation in the report.

**Corner case to consider**: What are the corner cases you want to analyse here?

**Minimum data**: how many observations, and what about their rotations,
are needed for the stacked system to actually be solvable? How do you define whether a system is solvable here? Include that in your report


In [16]:
def compute_pivot_calibration(F_ptr_list, F_cal_list):

    # 1. Check that the inputs are valid

    if len(F_ptr_list) != len(F_cal_list):
        raise ValueError(
            "F_ptr_list and F_cal_list must have the same length"
        )

    if len(F_ptr_list) < 2:
        raise ValueError(
            "At least 2 observations are required"
        )

    # 2. Convert each pair of poses into a relative pose
    #       F_rel = F_cal.inv() * F_ptr

    F_rel_list = []

    for F_ptr, F_cal in zip(F_ptr_list, F_cal_list):
        F_rel = F_cal.inv() * F_ptr
        F_rel_list.append(F_rel)

    # 3. Extract R_j and t_j

    R_list = []
    t_list = []

    for F_rel in F_rel_list:

        R_list.append(F_rel.R.matrix)

        t_list.append(
            np.array([
                F_rel.p.x,
                F_rel.p.y,
                F_rel.p.z
            ])
        )

    # 4. Build A and b
    #
    #       A_j = R_j - R_1
    #       b_j = t_1 - t_j

    R1 = R_list[0]
    t1 = t_list[0]

    A_blocks = []
    b_blocks = []

    for j in range(1, len(R_list)):

        A_blocks.append(R_list[j] - R1)
        b_blocks.append(t1 - t_list[j])

    A = np.vstack(A_blocks)
    b = np.concatenate(b_blocks)

    # 5. Check rank / solvability

    rank = np.linalg.matrix_rank(A)

    if rank < 3:
        raise ValueError(
            f"Pivot calibration is not solvable: rank(A) = {rank}, "
            "but rank 3 is required."
        )

    # 6. Solve A p_tip = b

    p_solution, residuals, _, _ = np.linalg.lstsq(
        A,
        b,
        rcond=None
    )

    p_tip = vct3(
        p_solution[0],
        p_solution[1],
        p_solution[2]
    )

    # 7. Estimate covariance

    residual_vector = A @ p_solution - b

    dof = A.shape[0] - A.shape[1]

    if dof > 0:

        sigma_squared = (
            residual_vector @ residual_vector
        ) / dof

        covariance = (
            sigma_squared *
            np.linalg.inv(A.T @ A)
        )

    else:

        covariance = np.full((3, 3), np.nan)

    # 8. Return p_tip and covariance

    return p_tip, covariance


p_tip_est, p_tip_cov = compute_pivot_calibration(
    F_ptr_list,
    F_cal_list
)

print("calibrated p_tip:", p_tip_est)

print(
    "estimated covariance (mm^2):\n",
    p_tip_cov
)

print(
    "estimated std-dev per axis (mm):",
    np.sqrt(np.diag(p_tip_cov))
)
    # 1. Check that the inputs are valid

    # 2. Convert each pair of poses into a relative pose
    #       F_rel = F_cal.inv() * F_ptr

    # 3. Extract R_j and t_j

    # 4. Build A and b
    #
    #       A_j = R_j - R_1
    #       b_j = t_1 - t_j

    # 5. Check rank / solvability

    # 6. Solve A p_tip = b

    # 7. Estimate covariance

    # 8. Return p_tip, covariance



p_tip_est, p_tip_cov = compute_pivot_calibration(F_ptr_list, F_cal_list)


calibrated p_tip: vct3(x=-0.29, y=0.47, z=150.67)
Internal Vector:
[[ -0.285]
 [  0.472]
 [150.672]]
estimated covariance (mm^2):
 [[ 0.406  0.086 -0.042]
 [ 0.086  0.141 -0.005]
 [-0.042 -0.005  0.114]]
estimated std-dev per axis (mm): [0.637 0.376 0.338]


In [17]:
print("calibrated p_tip:", p_tip_est)
print("estimated covariance (mm^2):\n", p_tip_cov)
print("estimated std-dev per axis (mm):", np.sqrt(np.diag(p_tip_cov)))


calibrated p_tip: vct3(x=-0.29, y=0.47, z=150.67)
Internal Vector:
[[ -0.285]
 [  0.472]
 [150.672]]
estimated covariance (mm^2):
 [[ 0.406  0.086 -0.042]
 [ 0.086  0.141 -0.005]
 [-0.042 -0.005  0.114]]
estimated std-dev per axis (mm): [0.637 0.376 0.338]


In [9]:
F_rel_test = F_cal_list[0].inv() * F_ptr_list[0]

print(type(F_rel_test.R))
print(dir(F_rel_test.R))

<class 'data_types.nominal_types.Rot'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_build_matrix', '_matrix', 'axis_angle', 'cayley', 'eye', 'from_quaternion', 'inv', 'matrix', 'normalize', 'rand', 'rand_deg', 'rand_xyz', 'rand_xyz_deg', 'to_quaternion', 'xyz', 'xyz_deg']


In [10]:
print(type(F_rel_test.R.matrix))
print(F_rel_test.R.matrix)

<class 'numpy.ndarray'>
[[ 0.816  0.01  -0.579]
 [ 0.471  0.569  0.674]
 [ 0.336 -0.822  0.459]]


In [12]:
t_test = F_rel_test.p

print(type(t_test))
print(dir(t_test))
print(t_test)

<class 'data_types.nominal_types.vct3'>
['__abs__', '__add__', '__annotate_func__', '__annotations_cache__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__mul__', '__ne__', '__neg__', '__new__', '__reduce__', '__reduce_ex__', '__replace__', '__repr__', '__rmul__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__sub__', '__subclasshook__', '__truediv__', '__weakref__', '_vec', 'cross', 'distance', 'dot', 'from_skew', 'homog', 'norm', 'null', 'rand', 'skew', 'times', 'unit', 'vec', 'x', 'y', 'z']
vct3(x=87.08, y=-101.80, z=-67.20)
Internal Vector:
[[  87.076]
 [-101.797]
 [ -67.205]]


## Step 8: Validate against test poses

> "**Hint**: You may want to consider placing additional marker bodies
> close to (but not inside) the cube defined by the test volume. However,
> this will also depend on your design for and placement of the
> calibration object."

The nominal test-pose grid is `theta_x, theta_y, theta_z` each in
`{0, +-pi/6, +-pi/3, +-pi/2}` and `p_x, p_y, p_z` each in
`{300, 300+-100, 300+-200}` mm. For each test pose, the assignment requires a
**single observation only** (no averaging).

Think about why the hint suggests an *additional*, nearby reference body?

In [18]:
near_local_positions = [
    vct3(-40, -40, -40),
    vct3( 40, -40,  40),
    vct3(-40,  40,  40),
    vct3( 40,  40, -40),
]

F_near_approx = Frame(
    Rot.eye(),
    vct3(600, 300, 300)
)

near_mb = simulation.define_marker_body(
    "near_mb",
    near_local_positions
)

near_placed = simulation.place_marker_body(
    near_mb,
    F_near_approx
)

refine_marker_body(trk, near_placed)

In [ ]:
near_local_positions = [
    # TODO: at least 4 non-coplanar vct3 points for this reference body
]
F_near_approx = None  # TODO: Frame, placed close to (not inside) the test-pose cube

near_mb = simulation.define_marker_body("near_mb", near_local_positions)
near_placed = simulation.place_marker_body(near_mb, F_near_approx)
refine_marker_body(trk, near_placed)


Refine this new body's geometry the same way you refined the pointer's and
calibration object's (Step 3b) before using it.


Now the single-observation test loop. For each nominal test pose `F_t`:
place the pointer there (small, robot-precision placement error, same as
the real system), take one simultaneous reading of the pointer and your
Step-8 reference body, and predict the tip's position (relative to that
reference body) using your calibrated `p_tip`.

**Ground truth for reporting only**: `PlacedBody.actual_placement` and
`MarkerBody.actual_marker_positions` hold the hidden values the simulator
sampled — you have access to them here only because this notebook doesn't
gate them behind a separate eval mode (per the handout: "During debugging
you will have access to all of the ... values. These will not be available
during evaluation."). Use them *only* to compute and report your achieved
error in this section — using them anywhere in your actual calibration math
(Steps 3b/7) kills the point of the assignment.


In [19]:
thetas = [0, np.pi / 6, -np.pi / 6, np.pi / 3, -np.pi / 3, np.pi / 2, -np.pi / 2]
offsets = [0, 100, -100, 200, -200]
# TODO: the full grid above is large (7^3 * 5^3 = 42,875 combinations) --
# choose a representative subset of (position, orientation) test poses
# rather than every combination.

# Representative subset of orientations
test_thetas = [
    (0, 0, 0),

    ( np.pi/6, 0, 0),
    (-np.pi/6, 0, 0),
    ( np.pi/3, 0, 0),
    (-np.pi/3, 0, 0),
    ( np.pi/2, 0, 0),
    (-np.pi/2, 0, 0),

    (0,  np.pi/6, 0),
    (0, -np.pi/6, 0),

    (0, 0,  np.pi/6),
    (0, 0, -np.pi/6),

    ( np.pi/6,  np.pi/6, 0),
    (0,  np.pi/6,  np.pi/6),
]

# Use the center and the six ±200 mm offsets.
# This gives positions throughout the test cube.
test_offsets = [-200, 0, 200]

test_poses = [
    # TODO: build a list of Frame(Rot.xyz(rx, ry, rz), vct3(300+tx, 300+ty, 300+tz))
    # from thetas/offsets above
]
for rx, ry, rz in test_thetas:

    for tx in test_offsets:
        for ty in test_offsets:
            for tz in test_offsets:

                F_t = Frame(
                    Rot.xyz(rx, ry, rz),
                    vct3(
                        300 + tx,
                        300 + ty,
                        300 + tz
                    )
                )

                test_poses.append(F_t)

print("number of test poses:", len(test_poses))


test_results = []  # list of (F_t_nominal, error_mm)
for F_t_nominal in test_poses:
    # 1. Place the pointer at the test pose.
    #    "precise" introduces the simulator's small placement error.
    test_placed = simulation.place_marker_body(
        ptr_mb,
        F_t_nominal,
        precision="precise"
    )

    # 2. Take ONE simultaneous tracker reading.
    obs = simulation.sample_marker_bodies(
        trk,
        [test_placed, near_placed]
    )

    # 3. Compute the observed pointer pose relative to
    #    the nearby reference body.
    F_rel_obs = (
        obs["near_mb"].frame.inv()
        * obs["ptr_mb"].frame
    )

    # 4. Predict the tip position relative to the
    #    nearby reference body using calibrated p_tip.
    p_pred = F_rel_obs * p_tip_est

    # 5. Ground truth -- ONLY for validation/reporting.
    F_true_rel = (
        near_placed.actual_placement.inv()
        * test_placed.actual_placement
    )

    p_true = F_true_rel * ptr_tip_approx

    # 6. Compute position error.
    error_vec = p_pred - p_true
    error_mm = error_vec.norm()

    # 7. Store the result.
    test_results.append(
        (F_t_nominal, error_mm)
    )

print("completed", len(test_results), "single-observation tests")
    # TODO, for each F_t_nominal:
    #   1. test_placed = simulation.place_marker_body(ptr_mb, F_t_nominal, precision="precise")
    #   2. obs = simulation.sample_marker_bodies(trk, [test_placed, near_placed])  -- single reading
    #   3. predict the tip position relative to near_placed using p_tip_est
    #   4. compute the TRUE tip position relative to near_placed using
    #      test_placed.actual_placement / near_placed.actual_placement (ground truth,
    #      for reporting only -- see markdown above)
    #   5. record the error and append (F_t_nominal, error) to test_results


number of test poses: 351
completed 351 single-observation tests


In [20]:
errors = np.array([
    error_mm
    for _, error_mm in test_results
])

mean_error = np.mean(errors)
max_error = np.max(errors)

within_spec = errors <= 2.0
n_within_spec = np.sum(within_spec)
n_total = len(errors)

print(f"number of test poses: {n_total}")
print(f"mean tip-position error: {mean_error:.3f} mm")
print(f"maximum tip-position error: {max_error:.3f} mm")
print(
    f"poses within 2 mm: "
    f"{n_within_spec}/{n_total} "
    f"({100 * n_within_spec / n_total:.1f}%)"
)

number of test poses: 351
mean tip-position error: 1.084 mm
maximum tip-position error: 2.287 mm
poses within 2 mm: 350/351 (99.7%)


In [21]:
worst_index = np.argmax(errors)

worst_pose, worst_error = test_results[worst_index]

print("worst test index:", worst_index)
print("worst error:", worst_error, "mm")

print("worst pose rotation:")
print(worst_pose.R.matrix)

print("worst pose translation:")
print(worst_pose.p)


worst test index: 350
worst error: 2.2869211225131956 mm
worst pose rotation:
[[ 0.75  -0.433  0.5  ]
 [ 0.5    0.866  0.   ]
 [-0.433  0.25   0.866]]
worst pose translation:
vct3(x=500.00, y=500.00, z=500.00)
Internal Vector:
[[500.]
 [500.]
 [500.]]


In [22]:
print("minimum error:", np.min(errors))
print("median error:", np.median(errors))
print("90th percentile:", np.percentile(errors, 90))
print("95th percentile:", np.percentile(errors, 95))
print("99th percentile:", np.percentile(errors, 99))

minimum error: 0.13025747520588768
median error: 1.0609133355068927
90th percentile: 1.5834063838372476
95th percentile: 1.69362063363621
99th percentile: 1.8656742042143855


## Achieved performance

Summarize the tip-position error across the test-pose grid against the
handout's 2mm spec.


In [ ]:
# TODO: summarize test_results -- e.g. mean/max error, and how many poses
# meet the assignment's ||delta p_t|| <= 2mm spec.
